# Notebook projet IMA segmentation d'images: Détection du ventricule gauche.

In [ ]:
# Chargement des librairies
import numpy as np
import matplotlib.pyplot as plt
import os

import plotly.graph_objects as go
import nibabel as nib

## Data analysis

Architecture:
- database
    - training
        - patient001
            - info.cfg
            - patient001_4d.nii.gz
            - patient001_frame01_gt.nii.gz
            - patient001_frame01.nii.gz
            - patient001_frame12_gt.nii.gz
            - patient001_frame12.nii.gz
        - ...
    - testing
        - patient101
            - ...
        - ...
        - patient150


In [ ]:
# View the distribution of the shapes of the images

# Load the images
images = []
base_path = 'database/training/'
# All training folders
folders = os.listdir(base_path)
print(f"Found {len(folders)} patients")

for folder in folders:
    # Load the 4d image
    image_path = os.path.join(base_path, folder, f'{folder}_4d.nii.gz')
    image = nib.load(image_path)
    image = image.get_fdata()
    # normalize the image
    image = (image - image.min()) / (image.max() - image.min())
    images.append(image)

# Plot the shape distribution in a 3d space
shapes = np.array([image.shape for image in images])
print(f"Xmin: {shapes[:, 0].min()}, Xmax: {shapes[:, 0].max()}")
print(f"Ymin: {shapes[:, 1].min()}, Ymax: {shapes[:, 1].max()}")
print(f"Minimum number of layers: {shapes[:, 2].min()}, Maximum number of layers: {shapes[:, 2].max()}")
print(f"Minimum number of time points: {shapes[:, 3].min()}, Maximum number of time points: {shapes[:, 3].max()}")

fig = go.Figure(data=[go.Scatter3d(x=shapes[:, 0], y=shapes[:, 1], z=shapes[:, 2], mode='markers', marker=dict(size=5))])
fig.update_layout(scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Layers'))
fig.show()

In [ ]:
# Visualisation des images d'un patient

training_path = "database/training"
validation_path = "database/validation"

# List all patients
patients = os.listdir(training_path)
print(patients)

# Visualisation des images d'un patient
numero_patient = 1

# Load the images
image_4d = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_4d.nii.gz")
image_01 = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame01.nii.gz")
image_01_gt = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame01_gt.nii.gz")
image_12 = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame12.nii.gz")
image_12_gt = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame12_gt.nii.gz")

print(image_4d.shape) # (216, 256, 10, 30) for patient 1
r, c, nb_frames, duration = image_4d.shape

print(image_01.shape) # (216, 256, 10) for patient 1
print(image_01_gt.shape) # (216, 256, 10) for patient 1

print(image_12.shape) # (216, 256, 10) for patient 1
print(image_12_gt.shape) # (216, 256, 10) for patient 1

# Plot a 2d slice, for time 0
slice = 1
plt.imshow(image_4d.dataobj[:,:,slice,0]) #,cmap='gray' for black and white
plt.show()

plt.imshow(image_01.dataobj[:,:,slice])
plt.show()

print(image_01_gt.shape) # (216, 256, 10) for patient 1
plt.imshow(image_01_gt.dataobj[:,:,9])
plt.show()

image_diff = image_12.dataobj[:,:,:] - image_01.dataobj[:,:,:]
# Seuillage
image_diff[abs(image_diff) < 0.1] = 0
for i in range(image_diff.shape[2]):
    plt.imshow(image_diff[:,:,i])
    plt.show()

In [ ]:
# Based on https://www.kaggle.com/code/abimbolasalawu/left-ventricle-using-cnn/notebook

# Visualisation des images d'un patient
numero_patient = 1

# Load the images
image_4d = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_4d.nii.gz")

print(image_4d.shape) # (216, 256, 10, 30)

# We want to visualize the 2D slices of the 3D volume in 3D space.
images = np.array(image_4d.get_fdata())

layer = 4
#images = images[:, :, layer, :] # to work with a given layer
images = images[:, :, :, layer] # to work with a given time frame
volume = images.T

nb_frames, r, c = volume.shape

fig = go.Figure(frames=[go.Frame(data=go.Surface(
    z=(((nb_frames-1)/10) - k * 0.1) * np.ones((r, c)),
    surfacecolor=np.flipud(volume[nb_frames-1 - k]),
    cmin=0, cmax=255
    ),
    name=str(k) # you need to name the frame for the animation to behave properly
    )
    for k in range(nb_frames)])

fig.add_trace(go.Surface(
    z=((nb_frames-1)/10) * np.ones((r, c)),
    surfacecolor=np.flipud(volume[nb_frames-1]),
    colorscale='Gray',
    cmin=0, cmax=255,
    colorbar=dict(thickness=20, ticklen=4)
    ))

def frame_args(duration):
    return {
            "frame": {"duration": duration},
            "mode": "immediate",
            "fromcurrent": True,
            "transition": {"duration": duration, "easing": "linear"},
        }

sliders = [
            {
                "pad": {"b": 10, "t": 60},
                "len": 0.9,
                "x": 0.1,
                "y": 0,
                "steps": [
                    {
                        "args": [[f.name], frame_args(0)],
                        "label": str(k),
                        "method": "animate",
                    }
                    for k, f in enumerate(fig.frames)
                ],
            }
        ]


# Layout
fig.update_layout(
         title='Slices in volumetric data',
         width=700,
         height=700,
         scene=dict(
                    zaxis=dict(range=[-0.1, nb_frames/10], autorange=False),
                    aspectratio=dict(x=1, y=1, z=1),
                    ),
         updatemenus = [
            {
                "buttons": [
                    {
                        "args": [None, frame_args(50)],
                        "label": "&#9654;", # play symbol
                        "method": "animate",
                    },
                    {
                        "args": [[None], frame_args(0)],
                        "label": "&#9724;", # pause symbol
                        "method": "animate",
                    },
                ],
                "direction": "left",
                "pad": {"r": 10, "t": 70},
                "type": "buttons",
                "x": 0.1,
                "y": 0,
            }
         ],
         sliders=sliders
)

fig.show()